# 04 - Semantic Analysis

## Intro

In questo capitolo andremo a sfruttare il machine learning per mappare prima il testo di ogni articolo, poi le singole parole in modo da poter analizzare e la somiglianza tematica e narrativa degli articoli e l'evoluzione del linguaggio (semantic shift).

In particolare la prima parte di dividerà in questo modo:
- Aggregazione del testo: unione di headline, abstract e lead_paragraph
- NLP processing: pulizia del testo 
- Vettorizzazione del testo
- Compressione t-SNE
- Rendering visivo


## 0. Setup

Controlliamo se abbiamo tutte le librerie e le dipendenze che ci servono e definiamo i path su cui andremo poi a lavorare.

In [ ]:
import importlib.util
import logging
import subprocess
import sys
from pathlib import Path

# import_name -> pip_name (spaCy/scikit-learn: nome import diverso dal nome pip)
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "nltk": "nltk",
    "spacy": "spacy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "gensim": "gensim",
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print(f"Installazione pacchetti mancanti: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Tutti i pacchetti richiesti sono gia' installati.")

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import spacy

%matplotlib inline

# Dati NLTK necessari per pulizia testo (tokenizzazione, stopword, lemmatizzazione)
for resource, path in [
    ("punkt", "tokenizers/punkt"),
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource, quiet=True)

# Modello spaCy inglese (corpus NYT) — se assente, download on-demand
SPACY_MODEL = "en_core_web_sm"
if not importlib.util.find_spec(SPACY_MODEL):
    print(f"Download modello spaCy '{SPACY_MODEL}'...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", SPACY_MODEL])
nlp = spacy.load(SPACY_MODEL)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"

IN_ICE = PROCESSED_DIR / "articles_ice.parquet"

sns.set_theme(style="whitegrid", palette="muted")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("semantic_analysis")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")


## 1. Aggregazione del testo 

In [ ]:
log.info(f"Caricamento dati da {IN_ICE.name}...")
df = pd.read_parquet(IN_ICE)

# Sostituisco i valori NaN con stringhe vuote per evitare errori di tokenizzazione
for col in ["headline", "abstract", "lead_paragraph"]:
    df[col] = df[col].fillna('')

df['semantic_text'] = (
    df['headline'] + ' ' +
    df['abstract'] + ' ' + 
    df['lead_paragraph'] 
    )

# Rimuovo eventuali spazi multipli e spazi iniziali/finali causati dalla concatenazione dei campi
df['semantic_text'] = df['semantic_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Esempio di visualizzazione dei primi articoli
print(f"Dataset caricato: {len(df)} articoli.")
print("Esempio di articoli (campo 'semantic_text'):")
print("-" * 50)
print(df[['semantic_text']].head(n=3))

## 2. Pulizia 

In [ ]:
clean_docs = []

for doc in nlp.pipe(df['semantic_text'], disable=["ner", "parser"]):
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    clean_docs.append(" ".join(tokens))

df['clean_text'] = clean_docs

tokens_prima = df['semantic_text'].apply(lambda x: len(x.split())).sum()
tokens_dopo = df['clean_text'].apply(lambda x: len(x.split())).sum()
riduzione = (1 - (tokens_dopo / tokens_prima)) * 100

print(f"\n--- COMPRESSIONE DATASET ---")
print(f"Volume originale:  {tokens_prima:,} parole")
print(f"Volume ripulito:   {tokens_dopo:,} lemmi")

print("\nPRIMA (Testo Aggregato):")
print(df['semantic_text'].iloc[0])

print("\nDOPO (Testo Pulito - Lemmatizzato):")
print(df['clean_text'].iloc[0])


## 3. Vettorizzazione 

Useremo la funzione di peso TF-IDF (Term Frequency-Inverse Document Frequency). 
Alla fine di questa fase ogni articolo sarà rappresentato come un vettore da 5000 dimensioni.

### TF-IDF (Term Frequency - Inverse Document Frequency)
È un indicatore statisticp che serve a misurare l importanza di una parola all'interno di un documento rispetto a un'intera libreria di documenti (corpus).
TF-IDF non premia le parole che appaiono più volte ma quelle cge sono altamente specifiche per un testo. In particolare l'indicatore è bilanciato da
- TF (Term Frequency) che misura quante volte una parola compare in un singolo articolo (più è ripetuta localmente più il suo punteggio sale).
- IDF (Inverse Document Frequency) misura quanto una parola è specifica a un documento (o quanto è comune a tutti gli altri) e agisce dunque come un penalizzatore per parole comuni in tutti i documenti.

Prima però avviene la vettorizzazione dei nostri documenti (colonna di testo) in una matrice documento-termine:
- L'algoritmo legge tutti gli articoli dall'inizio alla fine e crea un dizionario inserendoci ogni singola parola trovata (una colona ciascuna).
- Crea una matrice dove ogni riga è un articolo e ogni colonna una parola del dizionario
- Per ogni cella calcola TF-IDF: se la parola non è presente in quell'articolo avrà punteggio 0.0, se è presente più è specifica e più avrà punteggio alto.

### t-SNE
È un algoritmo che agisce dopo la vettorizzazione, prende la matrice dove ogni articolo è formato da 5000 dimensioni e schiaccia le coordinate in 2 dimensioni facendo in modo che articoli con coordinate a 5000 dimensioni simili (che quindi trattano temi simili/usano parole simili) rimangano vicini anche in 2 dimensioni:
- Per ogni articolo viene misurata la distanza da tutti gli altri articoli nello spazio a 5000 dimensioni: viene fatto creando una distribuzione gaussiana attorno all'articolo in modo che gli articoli simili cadono al centro e ottengono un punteggio di affinità alto.
- Crea la mappa 2D e inizialmente posiziona random tutti gli articoli nello spazio
- Ripete lo stesso calcolo di affinità di prima ma usando ora le coordinate 2D (usa la curva t-Student)
- Confronta i due punteggi e attira vicini articoli che originariamente erano vicini ma ora sono lontani e respinge viceversa
- Queste attrazioni e repulsioni vengono ripetute molte volte (diverse iterazioni) e alla fine i punti si assestano formando delle aree tematiche (parole usate simili)

### Perplexity
È un parametro critico dell algoritmo che indica il numero di vicini a cui ogni articolo presta attenzione: es. 5-10 ogni articolo guarda solo i vicinissimi e si formano tanti cluster piccoli; 50-100 ogni articolo guarda un'area molto vasta e si formano grandi cluster ma potremmo perdere delle sfumature. 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Configurazione del vettorizzatorein modo che:
# Ignora le parole che compaiono in più del 90% degli articoli (max_df=0.90)
# Ignora le parole che compaiono in meno di 3 articoli (min_df=3)
# Limita il numero di caratteristiche (parole) a 5000 (max_features=5000)
tfdif_vectorizer = TfidfVectorizer(
    max_features = 5000, 
    max_df = 0.90,
    min_df = 3
)

# Applico il vettorizzatore al campo 'clean_text' del DataFrame, creando una matrice TF-IDF
# .fit_transform() impara il vocabolario e trasforma i documenti in una matrice TF-IDF
tfidf_matrix = tfdif_vectorizer.fit_transform(df['clean_text'])

# Estraggo nomi dele parile che corrispondono alle colonne della matrice TF-IDF, che rappresentano le caratteristiche (features) del modello
feature_names = tfdif_vectorizer.get_feature_names_out()

print("\n--- RISULTATO VETTORIZZAZIONE ---")
print(f"Articoli processati: {tfidf_matrix.shape[0]}")
print(f"Dimensioni semantiche (parole uniche tenute): {tfidf_matrix.shape[1]}")
print(f"\nEsempio di parole estratte casualmente dal vocabolario:")
print(feature_names[1000:1010])  

## 4. Compressione spaziale

Andiamo ad usare l'algoritmo t-SNE per schiacciare i vettori da 5000 a 2 dimensioni (x,y) preservando però le caratteristiche semantiche degli articoli (se due articoli usano parole simili avranno coordinate simili).

Prima però di andare ad applicare t-SNE direttamente sulla matrice TF-IDF, applichiamo l'algoritmo Truncated SVD per ridurre la dimensione dei vettori da 5000 a 50 eliminando così il rumore di fonso e rendendo il calcolo di t-SNE più veloce e preciso.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_50d = svd.fit_transform(tfidf_matrix)

varianza_spiegata = svd.explained_variance_ratio_.sum()
print(f"Componenti: {tfidf_50d.shape[1]}")
print(f"Varianza spiegata cumulativa: {varianza_spiegata:.2%}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import pandas as pd

# 1. Prepariamo la colonna temporale e la nuova funzione per i periodi
df['pub_date'] = pd.to_datetime(df['pub_date'], utc=True)

def get_period(date):
    if date.year <= 2016: 
        return "2003-2016"
    elif date.year <= 2020: 
        return "2017-2020"
    elif date.year <= 2024: 
        return "2021-2024"
    else: 
        return "2025-Presente"

# Applichiamo la funzione per creare la colonna
df['periodo'] = df['pub_date'].apply(get_period)

# Aggiorniamo la palette collegandola ai nuovi nomi (4 colori ben distinti)
palette = {
    "2003-2016": "#1f77b4",     # Blu
    "2017-2020": "#d62728",     # Rosso
    "2021-2024": "#2ca02c",     # Verde
    "2025-Presente": "#9467bd"  # Viola
}

# 2. Definiamo i valori di perplexity da testare
perplexities = [10, 20, 30, 40, 50, 60]

# 3. Creiamo la griglia del grafico
fig, axarr = plt.subplots(nrows=3, ncols=2, figsize=(20, 18))
fig.subplots_adjust(hspace=0.3, wspace=0.2)

log.info("Avvio test grid search per t-SNE. Potrebbe richiedere qualche minuto...")

for i, ax in enumerate(axarr.reshape(-1)):
    p = perplexities[i]
    log.info(f"Calcolo t-SNE con perplexity={p}...")
    
    tsne = TSNE(n_components=2, perplexity=p, random_state=42, verbose=0)
    tsne_coords = tsne.fit_transform(tfidf_50d)
    
    sns.scatterplot(
        x=tsne_coords[:, 0], 
        y=tsne_coords[:, 1],
        hue=df['periodo'],     
        palette=palette,
        alpha=0.6,
        s=15,
        edgecolor=None,
        ax=ax,
        legend=(i == 0)        
    )
    
    ax.set_title(f"Perplexity = {p}", fontsize=14, fontweight='bold')
    ax.set_xticks([]) 
    ax.set_yticks([])
    
    if i == 0:
        ax.legend(title="Periodo", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.suptitle("Test Perplexity t-SNE sugli Abstract ICE", fontsize=20, y=0.92)
plt.show()


Guardando i grafici, emerge una dinamica classica dell'analisi testuale. Con valori bassi come 10 e 20, l'algoritmo frammenta eccessivamente lo spazio, creando piccole costellazioni sparse e rumorose. Man mano che la perplexity sale a 40, 50 e 60, la nuvola si condensa in una struttura globale molto più solida.

Il valore Perplexity = 50 offre il compromesso visivo migliore: mantiene la compattezza globale permettendo al contempo di distinguere bene le protuberanze e le densità locali.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# 1. Impostiamo il valore ottimale scelto visivamente
BEST_PERPLEXITY = 50
log.info(f"Calcolo t-SNE finale con perplexity={BEST_PERPLEXITY}...")

# 2. Ricalcoliamo il modello (verbose=1 per monitorare i tempi)
tsne_final = TSNE(n_components=2, perplexity=BEST_PERPLEXITY, random_state=42, verbose=1)
tsne_coords_final = tsne_final.fit_transform(tfidf_50d)

# 3. Salviamo le coordinate nel DataFrame in modo permanente
df['tsne_x'] = tsne_coords_final[:, 0]
df['tsne_y'] = tsne_coords_final[:, 1]

# 4. Rendering del grafico ad alta risoluzione
plt.figure(figsize=(16, 10))

sns.scatterplot(
    data=df, 
    x='tsne_x', 
    y='tsne_y', 
    hue='periodo', 
    palette=palette,  # Assicurati che il dizionario 'palette' della cella precedente sia caricato
    alpha=0.7,        # Leggermente più opaco per far risaltare i colori
    s=35,             # Pallini più grandi per il plot singolo
    edgecolor='white',
    linewidth=0.2
)

plt.title(f"Mappatura Semantica degli Articoli NYT sull'ICE\n(t-SNE, Perplexity={BEST_PERPLEXITY})", fontsize=18, pad=20)
plt.xlabel("Dimensione Semantica 1", fontsize=12)
plt.ylabel("Dimensione Semantica 2", fontsize=12)

# Spostiamo la legenda all'esterno del grafico per non coprire i dati
plt.legend(title="Periodo Storico", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, title_fontsize=14)
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()

log.info("Coordinate salvate con successo nel DataFrame (colonne 'tsne_x' e 'tsne_y').")

Il prossimo passo è l'Estrazione delle Keyword Dominanti.
Andremo a interrogare la nostra matrice matematica (il modello TF-IDF) per chiedergli: "Quali sono le 15 parole chiave che pesano di più matematicamente all'interno di ciascun periodo storico?"

In [ ]:
import numpy as np

log.info("Estrazione delle keyword dominanti per ogni periodo storico...")

# Assicuriamoci di riprendere il vocabolario dal nostro modello TF-IDF
feature_names = tfdif_vectorizer.get_feature_names_out()

# Quante parole vogliamo vedere per ogni epoca?
N_TOP_WORDS = 15

print("LE PAROLE CHIAVE DELL'ICE SUL NYT (Per Periodo Storico)\n" + "="*55)

# Cicliamo in ordine cronologico sui vari periodi
for periodo in sorted(df['periodo'].unique()):
    
    # 1. Troviamo gli indici di tutti gli articoli scritti in questo specifico periodo
    indici_periodo = np.where(df['periodo'] == periodo)[0]
    
    # 2. Estraiamo dalla matrice grande solo le righe (gli articoli) di questo periodo
    matrice_periodo = tfidf_matrix[indici_periodo]
    
    # 3. Calcoliamo l'importanza media (mean) di ogni parola in questo sotto-gruppo
    # Essendo una matrice sparsa, usiamo np.asarray per appiattirla in un vettore leggibile
    media_tfidf = np.asarray(matrice_periodo.mean(axis=0)).flatten()
    
    # 4. Troviamo le N parole con il punteggio più alto
    # argsort() ordina in ordine crescente, quindi prendiamo le ultime N e le rovesciamo ([::-1])
    top_indici = media_tfidf.argsort()[-N_TOP_WORDS:][::-1]
    
    # 5. Estraiamo le parole vere e proprie usando gli indici
    top_words = [feature_names[i] for i in top_indici]
    
    # Stampa formattata dei risultati
    print(f"\n PERIODO: {periodo} (Articoli: {len(indici_periodo)})")
    print("-" * 55)
    print(", ".join(top_words))

### 3.1 Distintività lessicale per periodo (log-odds con prior di Dirichlet)

La media TF-IDF della cella precedente premia le parole frequenti **in tutti** i periodi (*immigration, say, new, year*): dice poco su cosa distingue un periodo dagli altri.

Qui usiamo il **log-odds ratio con prior di Dirichlet informativo** (Monroe, Colaresi & Quinn 2008, *Fightin' Words*), lo standard per confrontare il lessico di due gruppi di testi:

- per ogni parola confronta la sua frequenza nel periodo con quella nel resto del corpus;
- il prior (le frequenze dell'intero corpus, peso `alpha0`) smorza le parole rare, che altrimenti avrebbero rapporti estremi per puro caso;
- il risultato è uno **z-score**: |z| > 1.96 ≈ differenza significativa al 5%.

Lavoriamo su `semantic_text` (non lemmatizzato), unigrammi e bigrammi: i bigrammi catturano formule come *illegal immigrants* vs *undocumented immigrants*, che la lemmatizzazione e i singoli token spezzerebbero.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

LOGODDS_STOPWORDS = list(ENGLISH_STOP_WORDS | {"said", "says", "mr", "ms", "new", "year", "years", "old"})
LOGODDS_ALPHA0 = 1000  # peso totale del prior: più alto = parole rare più penalizzate

logodds_vectorizer = CountVectorizer(
    stop_words=LOGODDS_STOPWORDS,
    ngram_range=(1, 2),
    min_df=5,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",  # esclude numeri e token di una lettera
)
counts_matrix = logodds_vectorizer.fit_transform(df['semantic_text'])
logodds_vocab = logodds_vectorizer.get_feature_names_out()


def log_odds_dirichlet(counts, mask_a, mask_b, alpha0=LOGODDS_ALPHA0) -> pd.DataFrame:
    """z-score del log-odds ratio (gruppo A vs gruppo B) con prior dalle frequenze dell'intero corpus."""
    y_a = np.asarray(counts[mask_a].sum(axis=0)).ravel()
    y_b = np.asarray(counts[mask_b].sum(axis=0)).ravel()
    y_all = np.asarray(counts.sum(axis=0)).ravel()
    prior = alpha0 * y_all / y_all.sum()
    n_a, n_b = y_a.sum(), y_b.sum()
    delta = (np.log((y_a + prior) / (n_a + alpha0 - y_a - prior))
             - np.log((y_b + prior) / (n_b + alpha0 - y_b - prior)))
    variance = 1 / (y_a + prior) + 1 / (y_b + prior)
    return pd.DataFrame({"term": logodds_vocab, "z": delta / np.sqrt(variance), "count_a": y_a, "count_b": y_b})


PERIODI = sorted(df['periodo'].unique())
N_DISTINCTIVE = 15

distinctive = {}
for periodo in PERIODI:
    mask = (df['periodo'] == periodo).values
    distinctive[periodo] = log_odds_dirichlet(counts_matrix, mask, ~mask).sort_values("z", ascending=False)

pd.DataFrame({
    periodo: [f"{r.term} ({r.z:.1f})" for r in distinctive[periodo].head(N_DISTINCTIVE).itertuples()]
    for periodo in PERIODI
})

**Come leggere la tabella.** Ogni colonna è un periodo; le righe sono le 15 parole o bigrammi più *caratteristici* di quel periodo rispetto a tutti gli altri, con lo z-score tra parentesi. Uno z-score alto non vuol dire che la parola sia la più frequente, ma che è **molto più frequente lì che altrove** (oltre ciò che ci si aspetterebbe dal caso).

**Cosa emerge.**
- **2003-2016**: il lessico dei *programmi* federali e della *criminalità* — *program, obama administration, sheriff, arizona, drug, convicted, deaths* — e l'etichetta **illegal immigrants** (z = 8,4).
- **2017-2020**: il lessico delle *famiglie* — *children, families, parents, separated, migrant children* — insieme a **undocumented immigrants** (z = 6,4), *sanctuary cities* e *wall*.
- **2021-2024**: il lessico del *confine* e dell'*asilo* — *asylum, southern border, guidelines, biden administration*.
- **2025-Presente**: il lessico dello *scontro sul territorio* — *crackdown, agents, judge, shooting, guard, protesters* e le città coinvolte (*minnesota, chicago*). La parola più distintiva in assoluto è *ice* (z = 17): in questo periodo il NYT usa la sigla invece del nome per esteso.

A differenza della media TF-IDF della cella precedente, le parole generiche (*say, new, year, immigration*) spariscono: restano solo quelle che distinguono davvero le epoche.

In [ ]:
fig, axes = plt.subplots(1, len(PERIODI), figsize=(22, 7), sharex=False)
for ax, periodo in zip(axes, PERIODI):
    top = distinctive[periodo].head(N_DISTINCTIVE).iloc[::-1]
    ax.barh(top['term'], top['z'], color=palette[periodo])
    ax.axvline(1.96, color='grey', linestyle=':', linewidth=1)
    ax.set_title(f"{periodo}\n({(df['periodo'] == periodo).sum()} articoli)", fontsize=13, fontweight='bold')
    ax.set_xlabel("z-score (periodo vs resto)")
plt.suptitle("Parole e bigrammi più distintivi di ogni periodo (log-odds, prior di Dirichlet)", fontsize=16)
plt.tight_layout()
plt.show()

**Come leggere il grafico.** Un pannello per periodo, con le stesse 15 parole della tabella. La lunghezza della barra è lo z-score; la linea tratteggiata grigia è la soglia 1,96 (significatività al 5%): tutte le barre la superano ampiamente, quindi nessuna di queste differenze è attribuibile al caso. Gli assi x non sono condivisi: la lunghezza delle barre va confrontata **dentro** un pannello, non tra pannelli (i periodi hanno dimensioni molto diverse e lo z-score dipende anche da quanti testi ci sono).

Da notare le barre del 2021-2024: più corte, perché il periodo ha solo 96 articoli e quindi meno evidenza statistica.

**Cosa cambia da un periodo al successivo.** Il confronto "periodo vs resto" dice cosa caratterizza un'epoca; il confronto tra periodi **consecutivi** isola invece le parole che entrano (z > 0) e quelle che escono (z < 0) nel passaggio da un'amministrazione all'altra.

In [ ]:
transizioni = list(zip(PERIODI[:-1], PERIODI[1:]))
righe = {}
for prima, dopo in transizioni:
    res = log_odds_dirichlet(counts_matrix, (df['periodo'] == dopo).values, (df['periodo'] == prima).values)
    res = res.sort_values("z")
    righe[f"{prima} → {dopo}: in crescita"] = [f"{r.term} ({r.z:.1f})" for r in res.tail(10).iloc[::-1].itertuples()]
    righe[f"{prima} → {dopo}: in calo"] = [f"{r.term} ({r.z:.1f})" for r in res.head(10).itertuples()]
pd.DataFrame(righe)

**Come leggere la tabella.** Per ogni passaggio tra periodi consecutivi, due colonne: le 10 parole che guadagnano più peso (*in crescita*, z > 0) e le 10 che ne perdono di più (*in calo*, z < 0). Qui il confronto è solo tra i due periodi adiacenti, non con tutto il corpus.

**Cosa emerge.**
- **2003-16 → 2017-20**: escono *obama, program, illegal, illegal immigrants*; entrano *trump, undocumented, undocumented immigrants, border, children*. Il cambio di etichetta per le persone è una delle variazioni più forti del passaggio.
- **2017-20 → 2021-24**: escono *children, parents, family, raids*; entrano *asylum, southern border, migrants*. L'attenzione si sposta dalle famiglie all'interno del paese al confine.
- **2021-24 → 2025+**: escono *border, asylum, migrants, undocumented immigrants*, ma anche *customs* e *customs enforcement*; entrano *ice, agents, crackdown, arrests* e le città (*minnesota, chicago*). Il racconto si sposta dal confine alle operazioni dentro le città americane, e il nome esteso dell'agenzia lascia il posto alla sigla.

**Come vengono nominate le persone.** Il confronto per periodo fa emergere un cambio di etichetta: *illegal immigrants* è distintivo del 2003-2016, *undocumented immigrants* del 2017-2020. Tracciamo anno per anno la quota di articoli che usa ciascuna formula (conteggio per articolo, non per occorrenza, così un singolo articolo che ripete il termine non pesa di più).

In [ ]:
ETICHETTE_PERSONE = {
    "illegal immigrant(s)": r"\billegal immigrants?\b",
    "illegal alien(s)": r"\billegal aliens?\b",
    "undocumented immigrant(s)": r"\bundocumented immigrants?\b",
    "migrant(s)": r"\bmigrants?\b",
}
testo_lower = df['semantic_text'].str.lower()
etichette = pd.DataFrame({nome: testo_lower.str.contains(rx, regex=True) for nome, rx in ETICHETTE_PERSONE.items()})
etichette['anno'] = df['pub_date'].dt.year
quota_annua = etichette.groupby('anno').mean() * 100
n_annuo = etichette.groupby('anno').size()
quota_annua = quota_annua[n_annuo >= 10]  # anni con meno di 10 articoli: quote troppo instabili

fig, ax = plt.subplots(figsize=(15, 6))
for nome in ETICHETTE_PERSONE:
    ax.plot(quota_annua.index, quota_annua[nome], marker='o', linewidth=2, label=nome)
for anno, testo in [(2017, 'Trump I'), (2021, 'Biden'), (2025, 'Trump II')]:
    ax.axvline(x=anno, color='grey', linestyle=':', alpha=0.7)
    ax.text(anno + 0.1, ax.get_ylim()[1] * 0.95, testo, color='grey', fontsize=10)
ax.set_title("Come il NYT nomina le persone negli articoli ICE: quota di articoli per anno (anni con ≥10 articoli)", fontsize=14)
ax.set_ylabel("% articoli dell'anno")
ax.set_xlabel("Anno")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Articoli per anno inclusi:", n_annuo[n_annuo >= 10].to_dict())

**Come leggere il grafico.** Ogni linea è una formula usata per nominare le persone; l'asse y è la **percentuale di articoli di quell'anno** che la contiene almeno una volta. Sono esclusi gli anni con meno di 10 articoli (2003-2007 e simili), dove una percentuale si baserebbe su 2-3 articoli. Le linee tratteggiate verticali segnano l'inizio delle amministrazioni. (Gli anni sull'asse x compaiono con decimali per un'impostazione di default di matplotlib: vanno letti come anni interi.)

**Cosa emerge.**
- **illegal immigrant(s)** (blu) è presente nel 17-30% degli articoli fino al 2012, crolla nel 2013 e sparisce del tutto dal 2015. Il crollo coincide con la revisione delle linee guida sul termine *illegal immigrant* adottata nel 2013 dall'Associated Press e, nello stesso periodo, dal NYT (da verificare e citare con la fonte nel report).
- **illegal alien(s)** (arancione) non viene mai usato in modo significativo: è un'etichetta presente nel dibattito politico ma non nel linguaggio del NYT.
- **undocumented immigrant(s)** (verde) diventa la formula di riferimento: picco nel 2017 (~30%) e di nuovo nel 2023 (~30%).
- **migrant(s)** (rosso) domina dal 2018 al 2024, con il picco nel 2022 (~44%), in corrispondenza della copertura sul confine e sull'asilo.
- **Nel 2025-2026 tutte le formule crollano** (sotto il 12% e poi sotto il 4%): il soggetto degli articoli non sono più le persone migranti ma l'agenzia e le sue operazioni (*agents, crackdown, raids*), coerente con la tabella di distintività del 2025-Presente.

## Word2Vec

In [ ]:
import logging
from gensim.models import Word2Vec

# 1. PREPARAZIONE DEI DATI
log.info("Preparazione delle frasi per Word2Vec...")
# Word2Vec richiede una "lista di liste" (es. [['ice', 'arrest'], ['border', 'patrol']])
# Usiamo .split() per trasformare le stringhe ripulite di nuovo in liste di token
sentences = [text.split() for text in df['clean_text'] if isinstance(text, str)]

log.info(f"Trovati {len(sentences)} articoli pronti per l'addestramento.")

# 2. ADDESTRAMENTO DEL MODELLO
log.info("Avvio addestramento di Word2Vec...")

# Parametri chiave del modello:
# vector_size=300 : ogni parola diventerà un vettore a 300 dimensioni (standard per testi complessi)
# window=5        : il "contesto". Guarda 5 parole prima e 5 parole dopo per capirne il significato
# min_count=5     : ignora le parole che compaiono meno di 5 volte in tutto il NYT
# workers=4       : usa 4 core del processore per velocizzare
# epochs=20       : fa leggere tutto il database all'algoritmo per 20 volte, migliorando l'apprendimento
w2v_model = Word2Vec(
    sentences=sentences, 
    vector_size=300, 
    window=5, 
    min_count=5, 
    workers=4,
    epochs=20
)

log.info("Modello addestrato con successo!")

# 3. IL PRIMO TEST: IL QUARTIERE SEMANTICO DI "ICE"
# Usiamo 'ice' in minuscolo perché in fase di pulizia NLP abbiamo abbassato tutti i caratteri
target_word = 'ice'

print(f"\nLE 20 PAROLE PIÙ 'VICINE' (SINONIMI CONTESTUALI) A '{target_word.upper()}':")
print("=" * 60)

try:
    # most_similar calcola la distanza matematica tra i vettori a 300 dimensioni
    similar_words = w2v_model.wv.most_similar(target_word, topn=20)
    for word, score in similar_words:
        # Il punteggio (score) va da 0 a 1 (1 = sinonimo perfetto)
        print(f"{word.ljust(20)} (Somiglianza: {score:.3f})")
except KeyError:
    print(f"Errore: la parola '{target_word}' non è presente nel vocabolario.")

### Analisi per epoca


In [ ]:
import logging
from gensim.models import Word2Vec
import pandas as pd

log.info("Inizio analisi diacronica Word2Vec per epoche...")

# Se non avevamo già la colonna, la ricreiamo per sicurezza
def get_period(date):
    if date.year <= 2016: return "2003-2016"
    elif date.year <= 2020: return "2017-2020"
    elif date.year <= 2024: return "2021-2024"
    else: return "2025-Presente"

if 'pub_date' in df.columns:
    df['pub_date'] = pd.to_datetime(df['pub_date'], utc=True)
    df['periodo'] = df['pub_date'].apply(get_period)

periodi = sorted(df['periodo'].unique())
target_word = 'ice'

# Dizionario per salvare i modelli addestrati in memoria
w2v_models = {}

print(f"\nEVOLUZIONE SEMANTICA DI '{target_word.upper()}' SUL NYT")
print("=" * 70)

for periodo in periodi:
    # 1. Filtriamo gli articoli dell'epoca
    df_periodo = df[df['periodo'] == periodo]
    
    # 2. Prepariamo le frasi
    sentences = [text.split() for text in df_periodo['clean_text'] if isinstance(text, str)]
    
    if len(sentences) == 0:
        continue
        
    log.info(f"Addestramento modello {periodo} su {len(sentences)} articoli...")
    
    # 3. Addestriamo il modello specifico per questa epoca
    # min_count=3 perché i corpus divisi sono più piccoli
    modello_epoca = Word2Vec(
        sentences=sentences, 
        vector_size=50, 
        window=7, 
        min_count=3,  
        workers=4,
        epochs=20
    )
    
    w2v_models[periodo] = modello_epoca
    
    # 4. Estraiamo i vicini semantici e li stampiamo su due colonne per leggibilità
    print(f"\n PERIODO: {periodo} (Articoli: {len(sentences)})")
    print("-" * 70)
    
    try:
        similar_words = modello_epoca.wv.most_similar(target_word, topn=14)
        
        for i in range(0, len(similar_words), 2):
            w1, s1 = similar_words[i]
            if i+1 < len(similar_words):
                w2, s2 = similar_words[i+1]
                print(f"{w1.ljust(15)} ({s1:.2f})  |  {w2.ljust(15)} ({s2:.2f})")
            else:
                print(f"{w1.ljust(15)} ({s1:.2f})")
    except KeyError:
        print(f"[!] La parola '{target_word}' non è presente in questo periodo.")

print("\n" + "=" * 70)
log.info("Analisi completata.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Assicuriamoci di avere l'anno per creare una linea continua
df['anno'] = df['pub_date'].dt.year

# 2. Scegliamo due gruppi di parole da confrontare nel tempo
# Puoi modificare queste liste con le parole che hai notato tu!
frame_istituzionale = ['customs', 'security', 'law', 'enforcement']
frame_conflittuale = ['protest', 'child', 'separated', 'activist']

parole_da_tracciare = frame_istituzionale + frame_conflittuale

# 3. Creiamo un dizionario per salvare i pesi medi annuali
dati_annuali = {parola: [] for parola in parole_da_tracciare}
anni = sorted(df['anno'].dropna().unique())

# 4. Estraiamo il peso TF-IDF di queste parole per ogni singolo anno
feature_list = list(feature_names) # Dal tuo TF-IDF precedente

for anno in anni:
    indici_anno = np.where(df['anno'] == anno)[0]
    if len(indici_anno) == 0:
        for p in parole_da_tracciare: dati_annuali[p].append(0)
        continue
        
    matrice_anno = tfidf_matrix[indici_anno]
    media_tfidf = np.asarray(matrice_anno.mean(axis=0)).flatten()
    
    for parola in parole_da_tracciare:
        if parola in feature_list:
            idx = feature_list.index(parola)
            dati_annuali[parola].append(media_tfidf[idx])
        else:
            dati_annuali[parola].append(0)

# 5. Trasformiamo i risultati in un DataFrame per il plot
df_trend = pd.DataFrame(dati_annuali, index=anni)

# 6. Generiamo il grafico delle traiettorie
plt.figure(figsize=(15, 8))

# Colori freddi per il burocratico, colori caldi per il conflittuale
colori = {
    'customs': '#add8e6', 'security': '#87ceeb', 'law': '#4682b4', 'enforcement': '#000080',
    'protest': '#ffb6c1', 'child': '#ff69b4', 'separated': '#dc143c', 'activist': '#8b0000'
}

for parola in parole_da_tracciare:
    if parola in frame_istituzionale:
        plt.plot(df_trend.index, df_trend[parola], label=f"{parola} (Istituzionale)", 
                 color=colori[parola], linestyle='--', linewidth=2)
    else:
        plt.plot(df_trend.index, df_trend[parola], label=f"{parola} (Conflittuale)", 
                 color=colori[parola], linewidth=3)

# Aggiungiamo le linee verticali per i cambi di presidenza
plt.axvline(x=2017, color='grey', linestyle=':', alpha=0.7)
plt.text(2017.2, plt.ylim()[1]*0.9, 'Inizio Era Trump', color='grey', fontsize=10)

plt.axvline(x=2021, color='grey', linestyle=':', alpha=0.7)
plt.text(2021.2, plt.ylim()[1]*0.9, 'Inizio Era Biden', color='grey', fontsize=10)

plt.title("Traiettorie del Vocabolario: Il Cambio di Framing sull'ICE (2003-2026)", fontsize=16)
plt.xlabel("Anno", fontsize=12)
plt.ylabel("Peso Semantico (TF-IDF)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd
import matplotlib.pyplot as plt

# 1. Definiamo la nostra blacklist di parole "standard" e vuote di significato specifico
parole_onnipresenti = [
    'ice', 'immigration', 'immigrant', 'immigrants', 'new', 'york', 
    'administration', 'say', 'said', 'trump', 'biden', 'obama', 
    'year', 'state', 'federal', 'official', 'government', 'president', 'agency',
    'states', 'city'
]

# 2. Configuriamo il vettorizzatore in modo molto più aggressivo
tf_vectorizer = CountVectorizer(
    max_df=0.50,               # NOVITÀ: ignora le parole presenti in più di metà del dataset
    min_df=5, 
    max_features=5000,
    stop_words=parole_onnipresenti  # NOVITÀ: applichiamo la blacklist
)

tf_matrix = tf_vectorizer.fit_transform(df['clean_text'])
tf_feature_names = tf_vectorizer.get_feature_names_out()

# 3. Addestriamo l'LDA (proviamo con 4 topic per vedere se si delineano meglio)
N_TOPICS = 4
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=15)
document_topics = lda.fit_transform(tf_matrix)

print("I 4 TOPIC LATENTI PURIFICATI DAL RUMORE DI FONDO:")
print("=" * 70)
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-11:-1]
    top_words = [tf_feature_names[i] for i in top_words_idx]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

## 5. Topic Modeling con BERTopic

Il TF-IDF (sezione 3) e il Word2Vec lavorano sulle singole parole. BERTopic invece raggruppa gli articoli per **significato della frase**, usando embedding di un modello transformer. Questo lo rende adatto a testi brevi come i nostri (headline + abstract + lead, ~50 parole per articolo), dove modelli classici come LDA faticano.

La pipeline di BERTopic:
1. **Embedding**: ogni articolo diventa un vettore da 384 dimensioni (modello `all-MiniLM-L6-v2`)
2. **UMAP**: riduzione a 5 dimensioni
3. **HDBSCAN**: clustering per densità; gli articoli che non stanno in nessun cluster finiscono nel topic `-1` (outlier)
4. **c-TF-IDF**: per ogni cluster estrae le parole più rappresentative, che diventano l'etichetta del topic

Usiamo `semantic_text` (non lemmatizzato): gli embedding funzionano meglio sulle frasi naturali che sui lemmi.

**Limite metodologico**: l'analisi è su titolo, abstract e lead paragraph, non sul testo integrale degli articoli.

### 5.1 Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

docs = df['semantic_text'].tolist()

log.info(f"Calcolo embedding con {EMBEDDING_MODEL} su {len(docs)} articoli...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedding_model.encode(docs, show_progress_bar=False)

print(f"Matrice embedding: {embeddings.shape[0]} articoli x {embeddings.shape[1]} dimensioni")

### 5.2 Addestramento

Parametri principali:
- `min_topic_size=15`: un topic deve contenere almeno 15 articoli
- `UMAP(random_state=42)`: senza seed fisso UMAP dà risultati diversi a ogni esecuzione
- `CountVectorizer(ngram_range=(1, 2))`: le etichette possono contenere bigrammi (es. "national guard", "sanctuary cities")

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP

# "ice" compare in quasi tutti gli articoli (come nel topic network, dove il nodo ICE è escluso):
# lasciarlo nelle etichette le renderebbe tutte uguali. Le altre sono parole di servizio del NYT
# ("old" viene dalle età, es. "34-year-old").
custom_stopwords = list(ENGLISH_STOP_WORDS | {"ice", "new", "said", "says", "mr", "ms", "year", "years", "old"})

vectorizer_model = CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2), min_df=3)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=15,
)

log.info("Addestramento BERTopic...")
topics, _ = topic_model.fit_transform(docs, embeddings)

n_outliers = sum(t == -1 for t in topics)
print(f"Topic trovati: {len(set(topics)) - 1}")
print(f"Articoli outlier (topic -1): {n_outliers} ({n_outliers / len(docs):.1%})")

### 5.3 Riduzione degli outlier

HDBSCAN è prudente: su testi brevi lascia fuori molti articoli. Con `reduce_outliers` assegniamo ogni outlier al topic più simile, ma solo se la similarità coseno tra il suo embedding e quello del topic supera una soglia (0.4). Sotto soglia l'articolo resta `-1`: forzare anche quelli significherebbe assegnare articoli che non somigliano davvero a nessun topic.

In [ ]:
OUTLIER_THRESHOLD = 0.4

new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD)
# Ricalcola le parole rappresentative sui topic aggiornati (altrimenti restano quelle pre-riassegnazione)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)

df['topic'] = new_topics

n_outliers_after = sum(t == -1 for t in new_topics)
print(f"Outlier prima:  {n_outliers} ({n_outliers / len(docs):.1%})")
print(f"Outlier dopo:   {n_outliers_after} ({n_outliers_after / len(docs):.1%})")

### 5.4 I topic trovati

Per ogni topic: numero di articoli e parole più rappresentative. Assegniamo anche un'etichetta breve (le prime 3 parole) da usare nei grafici.

In [ ]:
topic_info = topic_model.get_topic_info()

topic_labels = {
    row.Topic: f"{row.Topic}: " + ", ".join(w for w, _ in topic_model.get_topic(row.Topic)[:3])
    for row in topic_info.itertuples()
    if row.Topic != -1
}
topic_labels[-1] = "-1: outlier"
df['topic_label'] = df['topic'].map(topic_labels)

topic_table = topic_info[['Topic', 'Count']].copy()
topic_table['Parole rappresentative'] = topic_table['Topic'].apply(
    lambda t: ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
)
pd.set_option('display.max_colwidth', 120)
display(topic_table)

Leggere solo le parole non basta per interpretare un topic: vediamo i titoli degli articoli più rappresentativi di ciascuno.

In [ ]:
for t in sorted(topic_labels):
    if t == -1:
        continue
    print(f"\n{topic_labels[t]}  ({(df['topic'] == t).sum()} articoli)")
    print("-" * 70)
    for doc in topic_model.get_representative_docs(t)[:3]:
        print(f"  • {doc[:130]}...")

### 5.5 Parole principali per topic

In [ ]:
TOP_N_TOPICS = 12
top_topics = [t for t in topic_info['Topic'] if t != -1][:TOP_N_TOPICS]

fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
colors = sns.color_palette("muted", len(top_topics))

for ax, t, color in zip(axes.reshape(-1), top_topics, colors):
    words, scores = zip(*topic_model.get_topic(t)[:8])
    ax.barh(words[::-1], scores[::-1], color=color)
    ax.set_title(f"Topic {t} ({(df['topic'] == t).sum()} art.)", fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle("Parole più rappresentative per topic (c-TF-IDF)", fontsize=18)
plt.tight_layout()
plt.show()

### 5.6 Mappa semantica colorata per topic

Riprendiamo le coordinate t-SNE della sezione 4 e coloriamo gli articoli per topic. Se BERTopic e la mappa t-SNE (calcolata sul TF-IDF, un metodo indipendente) sono coerenti, gli articoli dello stesso topic dovrebbero formare zone compatte.

In [ ]:
plot_df = df[df['topic'].isin(top_topics)]

plt.figure(figsize=(16, 10))
sns.scatterplot(
    x=df['tsne_x'], y=df['tsne_y'],
    color='lightgrey', alpha=0.4, s=15, edgecolor=None, label='altri topic / outlier'
)
sns.scatterplot(
    data=plot_df, x='tsne_x', y='tsne_y',
    hue='topic_label', hue_order=[topic_labels[t] for t in top_topics],
    palette='tab20', alpha=0.8, s=30, edgecolor='white', linewidth=0.2
)
plt.title(f"Mappa t-SNE degli articoli ICE colorata per topic BERTopic (top {TOP_N_TOPICS})", fontsize=16)
plt.xticks([]); plt.yticks([])
plt.xlabel(""); plt.ylabel("")
plt.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

### 5.7 Composizione dei topic per periodo

I periodi hanno dimensioni molto diverse (il 2025-Presente da solo ha più articoli di tutti gli altri insieme), quindi i conteggi assoluti sarebbero dominati dall'ultimo periodo. Normalizziamo per periodo: ogni colonna mostra **che quota degli articoli di quel periodo** appartiene a ciascun topic.

In [ ]:
topic_period = pd.crosstab(df['topic_label'], df['periodo'], normalize='columns') * 100
topic_period = topic_period.drop(index=topic_labels[-1], errors='ignore')
topic_period = topic_period.loc[topic_period.max(axis=1).sort_values(ascending=False).index]

print("Articoli per periodo:")
print(df['periodo'].value_counts().sort_index().to_string())

plt.figure(figsize=(12, 14))
sns.heatmap(topic_period, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': '% articoli del periodo'})
plt.title("Peso di ciascun topic per periodo (%)", fontsize=16)
plt.xlabel("Periodo")
plt.ylabel("Topic")
plt.tight_layout()
plt.show()

### 5.8 Evoluzione dei topic nel tempo

Conteggio annuale degli articoli per i topic più grandi, con `topics_over_time` di BERTopic.

In [ ]:
df['anno'] = df['pub_date'].dt.year

topics_over_time = topic_model.topics_over_time(docs, df['anno'].tolist(), nr_bins=None)

# topics_over_time omette gli anni con 0 articoli: senza reindex la linea unirebbe
# direttamente gli anni non vuoti, facendo sembrare graduale un salto
anni_completi = range(df['anno'].min(), df['anno'].max() + 1)
trend = (
    topics_over_time[topics_over_time['Topic'].isin(top_topics[:8])]
    .pivot(index='Timestamp', columns='Topic', values='Frequency')
    .reindex(anni_completi, fill_value=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(16, 8))
colors = sns.color_palette('tab10', 8)
for t, color in zip(top_topics[:8], colors):
    ax.plot(trend.index, trend[t], marker='o', linewidth=2, color=color, label=topic_labels[t])

for anno, testo in [(2017, 'Inizio Era Trump I'), (2021, 'Inizio Era Biden'), (2025, 'Inizio Era Trump II')]:
    ax.axvline(x=anno, color='grey', linestyle=':', alpha=0.7)
    ax.text(anno + 0.1, ax.get_ylim()[1] * 0.95, testo, color='grey', fontsize=10)

ultimo_mese = df['pub_date'].max().strftime('%Y-%m')
ax.set_xticks(list(anni_completi))
ax.set_xticklabels(anni_completi, rotation=45)
ax.set_title(f"Evoluzione annuale dei principali topic ICE sul NYT (ultimo anno parziale, dati fino a {ultimo_mese})", fontsize=16)
ax.set_xlabel("Anno")
ax.set_ylabel("Numero di articoli")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 5.9 Gerarchia dei topic

Quali topic sono semanticamente vicini? L'albero mostra come BERTopic li fonderebbe se chiedessimo meno topic: i rami che si uniscono prima sono i temi più affini.

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
print(topic_model.get_topic_tree(hierarchical_topics))

### 5.10 Stabilità su più seed

UMAP è stocastico: con un seed diverso la mappa a 5 dimensioni cambia e HDBSCAN può trovare gruppi diversi. Sul Guardian (`07_guardianTopicModeling.ipynb`) il testo breve è risultato instabile: da 3 a 22 topic a seconda del seed, con 2 collassi su 5 in un unico topic gigante. Il modello qui sopra usa testo breve e un solo seed (42), quindi va verificato che i suoi topic non dipendano dal caso.

Riaddestriamo lo stesso modello, con gli stessi parametri, su `STABILITY_SEEDS` (i 5 del Guardian più altri 5) e misuriamo:

- **numero di topic** e **topic più grande** per seed (un topic oltre il 50% del corpus è un collasso);
- **AMI** (Adjusted Mutual Information) tra ogni coppia di seed: 1 = stessi gruppi, 0 = gruppi indipendenti. Calcolata due volte: su tutti gli articoli dopo la riassegnazione degli outlier, e solo sugli articoli che HDBSCAN ha messo in un cluster in entrambi i seed (il nucleo dei topic, senza gli articoli riassegnati).

Il modello di riferimento resta quello con seed 42 addestrato sopra: non viene riaddestrato.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_mutual_info_score

REFERENCE_SEED = 42
STABILITY_SEEDS = [0, 1, 7, 42, 123, 2, 3, 5, 11, 99]
OUT_TOPIC_STABILITY = PROCESSED_DIR / "ice_topic_stability.csv"

logging.getLogger("BERTopic").setLevel(logging.WARNING)


def fit_topics_seed(seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Stessa pipeline di 5.2-5.3 con un altro seed UMAP. Ritorna (topic HDBSCAN, topic dopo reduce_outliers)."""
    model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        vectorizer_model=CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2), min_df=3),
        min_topic_size=15,
    )
    raw, _ = model.fit_transform(docs, embeddings)
    raw = np.array(raw)
    final = raw
    if (raw == -1).any():
        final = np.array(model.reduce_outliers(docs, raw, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD))
    return raw, final


seed_runs = {REFERENCE_SEED: (np.array(topics), np.array(new_topics))}
for seed in STABILITY_SEEDS:
    if seed not in seed_runs:
        seed_runs[seed] = fit_topics_seed(seed)

seed_table = pd.DataFrame([
    {
        "seed": seed,
        "topic": len(set(final) - {-1}),
        "outlier prima %": 100 * (raw == -1).mean(),
        "outlier dopo %": 100 * (final == -1).mean(),
        "topic più grande %": 100 * pd.Series(final[final != -1]).value_counts().max() / len(final),
    }
    for seed, (raw, final) in ((s, seed_runs[s]) for s in STABILITY_SEEDS)
]).set_index("seed")
display(seed_table.round(1))

ami_all, ami_core = [], []
for a, b in combinations(STABILITY_SEEDS, 2):
    ami_all.append(adjusted_mutual_info_score(seed_runs[a][1], seed_runs[b][1]))
    core = (seed_runs[a][0] != -1) & (seed_runs[b][0] != -1)
    ami_core.append(adjusted_mutual_info_score(seed_runs[a][0][core], seed_runs[b][0][core]))

print(f"Topic per seed: {seed_table['topic'].min()}-{seed_table['topic'].max()} (riferimento, seed {REFERENCE_SEED}: {seed_table.loc[REFERENCE_SEED, 'topic']})")
print(f"Collassi (un topic > 50% del corpus): {(seed_table['topic più grande %'] > 50).sum()}/{len(STABILITY_SEEDS)}")
print(f"AMI tra seed, tutti gli articoli:     media {np.mean(ami_all):.3f} (min {min(ami_all):.3f}, max {max(ami_all):.3f})")
print(f"AMI tra seed, solo nucleo dei topic:  media {np.mean(ami_core):.3f} (min {min(ami_core):.3f}, max {max(ami_core):.3f})")

**Stabilità dei singoli topic del modello di riferimento.** L'AMI è una media su tutto il modello; qui guardiamo topic per topic. Per ogni topic del seed 42 cerchiamo, in ciascun altro seed, il topic che gli somiglia di più e misuriamo la sovrapposizione con l'indice di Jaccard (articoli in comune / articoli nell'uno o nell'altro: 1 = stesso insieme di articoli). Un topic con Jaccard medio alto ricompare quasi identico a ogni seed; uno basso viene spezzato o fuso in modo diverso ogni volta e va interpretato con cautela.

`riassegnati %` è la quota di articoli del topic che HDBSCAN aveva lasciato fuori e che `reduce_outliers` ha poi assegnato.

In [ ]:
STABLE_JACCARD = 0.7

ref_raw, ref_final = seed_runs[REFERENCE_SEED]
rows = []
for t in sorted(set(ref_final) - {-1}):
    members = np.where(ref_final == t)[0]
    member_set = set(members)
    scores = []
    for seed in STABILITY_SEEDS:
        if seed == REFERENCE_SEED:
            continue
        other = seed_runs[seed][1]
        best = 0.0
        for u in set(other[members]) - {-1}:
            other_set = set(np.where(other == u)[0])
            best = max(best, len(member_set & other_set) / len(member_set | other_set))
        scores.append(best)
    rows.append({
        "topic": topic_labels[t],
        "articoli": len(members),
        "riassegnati %": round(100 * (ref_raw[members] == -1).mean()),
        "jaccard medio": round(float(np.mean(scores)), 3),
        "jaccard minimo": round(float(min(scores)), 2),
        "stabile": np.mean(scores) >= STABLE_JACCARD,
    })

topic_stability = pd.DataFrame(rows).sort_values("jaccard medio", ascending=False).reset_index(drop=True)
topic_stability.to_csv(OUT_TOPIC_STABILITY, index=False)
log.info("Scritto %s", OUT_TOPIC_STABILITY.relative_to(BASE_DIR))

n_stable = int(topic_stability["stabile"].sum())
print(f"Topic stabili (Jaccard medio >= {STABLE_JACCARD}): {n_stable}/{len(topic_stability)}, "
      f"{topic_stability.loc[topic_stability['stabile'], 'articoli'].sum() / len(df):.0%} degli articoli")
display(topic_stability)

**Lettura (run 2026-09-30).**

- **Il modello NYT è stabile.** Su 10 seed: 23-28 topic, nessun collasso, AMI medio 0,80. Sul nucleo dei topic (articoli che HDBSCAN mette in un cluster) l'AMI sale a 0,96: i gruppi di base sono quasi identici a ogni seed. L'instabilità trovata sul testo breve del Guardian (AMI 0,39, 2 collassi su 5) qui non si ripresenta.
- **La variabilità viene dagli articoli riassegnati.** Il 36-45% degli articoli è outlier prima di `reduce_outliers`; sono loro a cambiare topic tra un seed e l'altro, e a far scendere l'AMI da 0,96 a 0,80.
- **Il seed 42 è il più prudente**: 23 topic, il minimo tra i 10 seed. Gli altri seed spezzano in più parti alcuni topic grandi.
- **18 topic su 23 sono stabili** (Jaccard medio ≥ 0,7) e coprono il 69% degli articoli. Da interpretare con cautela: **5** (giudici e tribunali, 0,40: il meno affidabile), **1** (Trump e deportazioni, 0,63), **15** (bambini e famiglie migranti, 0,66, con il 65% di articoli riassegnati), **18** e **19** (appena sotto soglia). I topic 1 e 5 sono grandi e generici: a seconda del seed vengono divisi o fusi con i vicini.
- I topic legati a un evento o a un luogo preciso (Guardia Nazionale, Guantánamo, proteste a Los Angeles, città santuario) sono i più solidi.

Limiti: la soglia 0,7 è una convenzione; il Jaccard minimo mostra che anche un topic stabile in media può sparire in un singolo seed (topic 20, studenti: 0,06 in un seed su 9).